# Caso de estudio: Predicción del consumo de golosinas en jóvenes con Árbol de Decisión

## Objetivo de aprendizaje
En este ejercicio aplicaremos un **modelo predictivo de Árbol de Decisión para regresión** para estimar el consumo semanal de golosinas en jóvenes.

Al finalizar, el estudiante debería ser capaz de:

- identificar la **variable objetivo**
- identificar las **variables predictoras**
- cargar un dataset en Google Colab
- dividir los datos en **entrenamiento** y **prueba**
- entrenar un modelo de **Árbol de Decisión**
- interpretar cómo toma decisiones el modelo
- evaluar el modelo con **métricas**
- concluir si la técnica aplicada es adecuada o no

---

## Contexto del caso
Una cadena de kioscos cercana a establecimientos educacionales quiere estimar cuántos paquetes de golosinas consume un joven por semana.

Para ello, dispone de variables como:
- edad
- dinero semanal disponible
- horas de estudio
- horas de actividad física
- uso de redes sociales
- salidas semanales
- distancia al kiosco
- promociones vistas

La variable que se desea predecir es:

**CONSUMO_GOLOSINAS_PAQUETES_SEMANA**

Como esta variable es **numérica**, seguimos frente a un problema de **regresión**.  
En esta oportunidad, en vez de usar regresión lineal, utilizaremos un **Árbol de Decisión para regresión**.


## Paso 1: Cargar el dataset

En Google Colab primero debemos subir el archivo CSV.

Ejecuta la siguiente celda y selecciona el archivo:

**dataset_consumo_golosinas_jovenes.csv**


In [ ]:
from google.colab import files
uploaded = files.upload()

## Paso 2: Importar librerías

Estas librerías nos permitirán:
- manipular datos con **pandas**
- hacer cálculos con **numpy**
- dividir datos con **train_test_split**
- entrenar el modelo con **DecisionTreeRegressor**
- evaluar el desempeño con métricas
- visualizar resultados con **matplotlib**
- representar el árbol de forma gráfica


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Paso 3: Leer el archivo CSV

Ahora cargamos el dataset y mostramos sus primeras filas para verificar que los datos se hayan leído correctamente.


In [ ]:
df = pd.read_csv("dataset_consumo_golosinas_jovenes.csv")

print("Dimensión del dataset:", df.shape)
df.head()

## Paso 4: Comprender las variables del problema

### Variable objetivo
Es la variable que queremos estimar:

- **CONSUMO_GOLOSINAS_PAQUETES_SEMANA**

### Variables predictoras
Son las variables que usaremos para explicar o predecir el consumo:

- EDAD
- DINERO_SEMANAL_CLP
- ACTIVIDAD_FISICA_HORAS
- HORAS_ESTUDIO_SEMANA
- HORAS_REDES_SOCIALES_DIA
- SALIDAS_SEMANA
- DISTANCIA_KIOSCO_KM
- PROMOCIONES_VISTAS_SEMANA


In [ ]:
print("Columnas del dataset:")
for c in df.columns:
    print("-", c)

## Paso 5: Revisión general de los datos

Aquí verificamos:
- tipos de datos
- si existen valores nulos
- estadísticas generales

Esto es importante porque un modelo no puede aprender correctamente si los datos están mal cargados o incompletos.


In [ ]:
print("Información general:")
print(df.info())

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nResumen estadístico:")
df.describe()

## Paso 6: Análisis exploratorio simple

Antes de modelar, conviene observar el comportamiento general de la variable objetivo.


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["CONSUMO_GOLOSINAS_PAQUETES_SEMANA"], bins=10)
plt.title("Distribución del consumo semanal de golosinas")
plt.xlabel("Paquetes por semana")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["DINERO_SEMANAL_CLP"], df["CONSUMO_GOLOSINAS_PAQUETES_SEMANA"])
plt.title("Dinero semanal vs consumo de golosinas")
plt.xlabel("Dinero semanal (CLP)")
plt.ylabel("Consumo de golosinas")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["PROMOCIONES_VISTAS_SEMANA"], df["CONSUMO_GOLOSINAS_PAQUETES_SEMANA"])
plt.title("Promociones vistas vs consumo de golosinas")
plt.xlabel("Promociones vistas por semana")
plt.ylabel("Consumo de golosinas")
plt.show()

## Paso 7: Separar variable objetivo y variables predictoras

En aprendizaje supervisado debemos separar:

- **X**: variables predictoras
- **y**: variable objetivo


In [ ]:
X = df.drop("CONSUMO_GOLOSINAS_PAQUETES_SEMANA", axis=1)
y = df["CONSUMO_GOLOSINAS_PAQUETES_SEMANA"]

print("Variables predictoras (X):")
print(X.head())

print("\nVariable objetivo (y):")
print(y.head())

## Paso 8: ¿Por qué usamos un Árbol de Decisión?

La técnica aplicada en este caso también es correcta porque:

1. El problema sigue siendo **predictivo**.
2. La variable objetivo sigue siendo **numérica**, por lo que necesitamos un modelo de **regresión**.
3. Un árbol de decisión divide los datos en grupos más homogéneos mediante reglas del tipo:
   - si ocurre esto, entonces predecir cierto valor
4. A diferencia de la regresión lineal, **no asume una relación lineal** entre las variables.
5. Puede capturar relaciones más complejas y más fáciles de explicar como reglas.

### Idea del modelo
Un árbol de decisión aprende preguntas del tipo:

- ¿DINERO_SEMANAL_CLP es mayor que cierto valor?
- ¿SALIDAS_SEMANA es mayor que cierto valor?
- ¿DISTANCIA_KIOSCO_KM es menor que cierto valor?

Según las respuestas, el modelo avanza por distintas ramas hasta llegar a una predicción final.


## Paso 9: Dividir datos en entrenamiento y prueba

No se debe entrenar y evaluar con los mismos datos.

Por eso el dataset se divide en:
- **entrenamiento**: para que el modelo aprenda
- **prueba**: para verificar qué tan bien predice en datos que nunca vio

Usaremos:
- 80% para entrenamiento
- 20% para prueba


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Cantidad de datos de entrenamiento:", X_train.shape[0])
print("Cantidad de datos de prueba:", X_test.shape[0])

## Paso 10: Entrenar el modelo

Aquí el algoritmo aprende reglas de decisión a partir de los datos.

En este ejemplo usamos:
- **max_depth = 4**

Esto limita la profundidad del árbol para evitar que memorice demasiado los datos de entrenamiento.


In [ ]:
modelo_arbol = DecisionTreeRegressor(max_depth=4, random_state=42)
modelo_arbol.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

## Paso 11: Importancia de las variables

En los árboles de decisión no interpretamos coeficientes como en la regresión lineal.

En cambio, podemos revisar la **importancia de cada variable**, es decir, cuánto aportó cada una a las decisiones del árbol.


In [ ]:
importancias = pd.DataFrame({
    "VARIABLE": X.columns,
    "IMPORTANCIA": modelo_arbol.feature_importances_
})

importancias.sort_values(by="IMPORTANCIA", ascending=False)

## Paso 12: Visualizar el árbol

Este gráfico muestra cómo el modelo va tomando decisiones paso a paso.

Cada nodo representa una regla.  
Cada rama representa una decisión posible.  
Cada hoja representa una predicción final.


In [ ]:
plt.figure(figsize=(22,10))
plot_tree(
    modelo_arbol,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árbol de decisión para predecir consumo de golosinas")
plt.show()

## Paso 13: Realizar predicciones

Ahora el modelo usará lo aprendido para estimar el consumo sobre el conjunto de prueba.


In [ ]:
y_pred = modelo_arbol.predict(X_test)

comparacion = pd.DataFrame({
    "VALOR_REAL": y_test.values,
    "VALOR_PREDICHO": np.round(y_pred, 2)
})

comparacion.head(10)

## Paso 14: Evaluar si el modelo está operando correctamente

Para saber si el modelo está funcionando bien, debemos medir su desempeño con métricas.

### Métricas utilizadas

#### 1. MAE (Mean Absolute Error)
Mide el error promedio en valor absoluto.  
Mientras más bajo, mejor.

#### 2. MSE (Mean Squared Error)
Mide el error al cuadrado.  
Penaliza más los errores grandes.

#### 3. RMSE (Root Mean Squared Error)
Es la raíz del MSE.  
Permite interpretar el error en la misma unidad que la variable objetivo.

#### 4. R² (Coeficiente de determinación)
Indica qué proporción de la variabilidad del consumo es explicada por el modelo.

- cercano a **1** → muy buen ajuste
- cercano a **0** → explica poco
- negativo → modelo muy deficiente

Estas métricas nos permiten justificar si el algoritmo está aplicando correctamente o no.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MÉTRICAS DEL MODELO")
print("-------------------")
print("MAE :", round(mae, 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2  :", round(r2, 2))

## Paso 15: Interpretación guiada de las métricas

El siguiente bloque entrega una interpretación automática básica para apoyar la discusión en clase.


In [ ]:
print("INTERPRETACIÓN GENERAL")
print("----------------------")

if r2 >= 0.75:
    print("El modelo presenta un ajuste alto.")
elif r2 >= 0.50:
    print("El modelo presenta un ajuste aceptable o moderado.")
elif r2 >= 0.30:
    print("El modelo presenta un ajuste básico: capta parte del fenómeno, pero aún puede mejorar.")
else:
    print("El modelo presenta un ajuste débil: no explica suficientemente el comportamiento del consumo.")

print("\nLectura adicional:")
print("- Un MAE bajo indica que, en promedio, la predicción se aleja poco del valor real.")
print("- Un RMSE alto indica presencia de errores más grandes en algunos casos.")
print("- El R2 muestra qué tan bien las variables escogidas logran explicar el consumo.")
print("- En árboles de decisión, un resultado muy alto también debe revisarse con cuidado para evitar sobreajuste.")

## Paso 16: Visualizar valores reales vs predichos

Este gráfico ayuda a verificar visualmente el comportamiento del modelo.

Si el modelo fuera perfecto, los puntos quedarían muy cerca del valor real esperado.


In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test, y_pred)
plt.title("Comparación entre valores reales y valores predichos")
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.show()

## Paso 17: Predicción de un nuevo caso

Ahora aplicaremos el modelo a un joven nuevo que no estaba en el dataset.

Esto demuestra la utilidad práctica del modelo predictivo.


In [ ]:
nuevo_joven = pd.DataFrame([{
    "EDAD": 16,
    "DINERO_SEMANAL_CLP": 18000,
    "ACTIVIDAD_FISICA_HORAS": 3,
    "HORAS_ESTUDIO_SEMANA": 12,
    "HORAS_REDES_SOCIALES_DIA": 5,
    "SALIDAS_SEMANA": 4,
    "DISTANCIA_KIOSCO_KM": 0.7,
    "PROMOCIONES_VISTAS_SEMANA": 6
}])

prediccion_nueva = modelo_arbol.predict(nuevo_joven)

print("Predicción para nuevo joven:")
print("Consumo estimado de golosinas por semana:", round(prediccion_nueva[0], 2))

## Evaluación complementaria: métricas de clasificación

Hasta este punto, el problema se ha trabajado como **regresión**, porque la variable objetivo es numérica.  
Sin embargo, para fines didácticos también podemos construir una **lectura complementaria de clasificación**.

### ¿Qué haremos?
Convertiremos el consumo semanal en una categoría binaria:

- **1 = Alto consumo**
- **0 = Bajo o consumo normal**

Para ello, usaremos como umbral el **promedio** del consumo observado en el dataset.

### ¿Por qué hacemos esto?
Porque métricas como:
- **precisión**
- **recall**
- **F1-score**
- **matriz de confusión**
- **AUC-ROC**

corresponden a problemas de **clasificación**, no de regresión.

Por lo tanto, este bloque debe entenderse como una **evaluación complementaria** del mismo caso, transformándolo en una decisión binaria:

> “¿El joven presenta alto consumo de golosinas o no?”

Esto permite que los estudiantes comparen ambos enfoques sobre el mismo dataset.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# -------------------------------------------------------
# 1. Definir umbral para convertir el problema en binario
# -------------------------------------------------------
umbral_alto_consumo = df["CONSUMO_GOLOSINAS_PAQUETES_SEMANA"].mean()

# Valores reales binarios
y_test_bin = (y_test >= umbral_alto_consumo).astype(int)

# Predicciones binarias a partir del modelo actual
y_pred_bin = (pd.Series(y_pred, index=y_test.index) >= umbral_alto_consumo).astype(int)

print("Umbral de alto consumo:", round(umbral_alto_consumo, 2))
print("\nInterpretación de clases:")
print("0 = Bajo o consumo normal")
print("1 = Alto consumo")

In [ ]:
# -------------------------------------------------------
# 2. Calcular métricas de clasificación
# -------------------------------------------------------
accuracy = accuracy_score(y_test_bin, y_pred_bin)
precision = precision_score(y_test_bin, y_pred_bin, zero_division=0)
recall = recall_score(y_test_bin, y_pred_bin, zero_division=0)
f1 = f1_score(y_test_bin, y_pred_bin, zero_division=0)
auc = roc_auc_score(y_test_bin, y_pred)

print("MÉTRICAS COMPLEMENTARIAS DE CLASIFICACIÓN")
print("----------------------------------------")
print("Accuracy :", round(accuracy, 2))
print("Precision:", round(precision, 2))
print("Recall   :", round(recall, 2))
print("F1-score :", round(f1, 2))
print("AUC-ROC  :", round(auc, 2))

### ¿Cómo interpretar estas métricas?

- **Accuracy**: proporción total de clasificaciones correctas.
- **Precision**: de todos los casos predichos como “alto consumo”, cuántos realmente lo eran.
- **Recall**: de todos los casos reales de “alto consumo”, cuántos logró detectar el modelo.
- **F1-score**: equilibrio entre precision y recall.
- **AUC-ROC**: mide la capacidad del modelo para separar ambas clases.

> Mientras más cercanos a **1**, mejor será el comportamiento del modelo en esta lectura binaria.

In [ ]:
# -------------------------------------------------------
# 3. Matriz de confusión
# -------------------------------------------------------
matriz = confusion_matrix(y_test_bin, y_pred_bin)
print("Matriz de confusión:")
print(matriz)

disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=["Bajo/Normal", "Alto"])
disp.plot()
plt.title("Matriz de confusión - evaluación complementaria")
plt.show()

In [ ]:
# -------------------------------------------------------
# 4. Curva ROC
# -------------------------------------------------------
RocCurveDisplay.from_predictions(y_test_bin, y_pred)
plt.title("Curva ROC - evaluación complementaria")
plt.show()

## Cierre de la evaluación complementaria

Este bloque no reemplaza la evaluación principal de regresión.  
Su propósito es ampliar la comprensión del caso y mostrar cómo, sobre el mismo problema, también podemos responder una pregunta distinta:

- **Regresión** → ¿Cuánto consumirá?
- **Clasificación** → ¿Presenta alto consumo o no?

Esta comparación ayuda a que los estudiantes comprendan que las métricas dependen del **tipo de problema formulado**.

## Paso 18: Conclusión del ejercicio

### ¿Por qué esta técnica es adecuada?
El **Árbol de Decisión para regresión** es apropiado en este caso porque:

- el problema es **predictivo**
- la variable objetivo es **numérica**
- puede modelar relaciones no lineales
- permite visualizar reglas de decisión
- entrega métricas claras para validar su funcionamiento

### ¿Cómo sabemos si el modelo está operando correctamente?
Lo evaluamos mediante:
- predicciones sobre datos no vistos
- comparación entre valor real y valor predicho
- métricas como MAE, MSE, RMSE y R²
- revisión de la importancia de variables
- visualización del árbol aprendido

### Idea final
No basta con que el código funcione.  
En minería de datos, un modelo se considera útil cuando:
1. responde al problema correcto
2. usa una técnica adecuada
3. se evalúa con métricas
4. permite interpretar resultados


## Preguntas para los estudiantes

1. ¿Cuál es la variable objetivo del problema?
2. ¿Por qué este caso corresponde a un modelo predictivo?
3. ¿Por qué un árbol de decisión también es una técnica adecuada?
4. ¿Qué ventaja tiene el árbol frente a la regresión lineal?
5. ¿Qué variables fueron más importantes para el modelo?
6. ¿Qué indica el valor de R² obtenido?
7. ¿El modelo parece suficientemente útil o debería mejorarse?
8. ¿Qué podría pasar si el árbol fuera demasiado profundo?
